# Compute F_j
(F_j) is the public transport service frequency at stop (j), measured as the number of scheduled arrivals per hour derived from GTFS data for randomly chosen day (1st weekday in the feed), for the time window between 7:00-9:00 am. 

In [9]:
from pathlib import Path
import pandas as pd
from datetime import date, timedelta


In [10]:
# =========================
# SETTINGS
# =========================

P1_ROOT = Path.cwd()  
DATA_DIR     = P1_ROOT / "data"


GTFS_DIR   = next(DATA_DIR.glob("gtfs-*"))  
START_TIME = "07:00:00"
END_TIME   = "09:00:00"
OUTPUT_CSV = DATA_DIR / "stops.csv"


def time_to_seconds(t):
    if pd.isna(t):
        return None
    h, m, s = str(t).split(":")
    return int(h)*3600 + int(m)*60 + int(s)

def yyyymmdd_to_date(s):
    return date(int(s[0:4]), int(s[4:6]), int(s[6:8]))

print(GTFS_DIR)
print(OUTPUT_CSV)


/Users/fgsumer/Desktop/thesis_code/Phase 1/data/gtfs-20260204-20260430
/Users/fgsumer/Desktop/thesis_code/Phase 1/data/stops.csv


In [11]:
# =========================
# Load GTFS
# =========================
stops = pd.read_csv(GTFS_DIR / "stops.txt", dtype=str)
stop_times = pd.read_csv(GTFS_DIR / "stop_times.txt", dtype=str)
trips = pd.read_csv(GTFS_DIR / "trips.txt", dtype=str)

calendar = pd.read_csv(GTFS_DIR / "calendar.txt", dtype=str)
caldates_path = GTFS_DIR / "calendar_dates.txt"
caldates = pd.read_csv(caldates_path, dtype=str) if caldates_path.exists() else None


# Determine 1st weekday in the feed

feed_start = yyyymmdd_to_date(calendar["start_date"].min())
feed_end   = yyyymmdd_to_date(calendar["end_date"].max())

first_weekday = None
d = feed_start
while d <= feed_end:
    if d.weekday() <= 4:  # Monday–Friday
        first_weekday = d
        break
    d += timedelta(days=1)

if first_weekday is None:
    raise ValueError("No weekday found in feed date range.")

print("Using weekday:", first_weekday)


# Determine active services on that specific date

active_services = set()

for _, r in calendar.iterrows():
    sid = r["service_id"]
    s0 = yyyymmdd_to_date(r["start_date"])
    s1 = yyyymmdd_to_date(r["end_date"])

    if not (s0 <= first_weekday <= s1):
        continue

    dow = first_weekday.weekday()
    if (
        (dow == 0 and r["monday"] == "1") or
        (dow == 1 and r["tuesday"] == "1") or
        (dow == 2 and r["wednesday"] == "1") or
        (dow == 3 and r["thursday"] == "1") or
        (dow == 4 and r["friday"] == "1")
    ):
        active_services.add(sid)

# Apply calendar_dates exceptions
if caldates is not None:
    caldates["date_obj"] = caldates["date"].apply(yyyymmdd_to_date)
    exceptions = caldates[caldates["date_obj"] == first_weekday]

    for _, r in exceptions.iterrows():
        sid = r["service_id"]
        if r["exception_type"] == "1":  # added
            active_services.add(sid)
        elif r["exception_type"] == "2":  # removed
            active_services.discard(sid)

if not active_services:
    raise ValueError("No active services found on selected weekday.")



Using weekday: 2026-02-04


In [12]:
# =========================
# Compute ARRIVALS in time window
# =========================

start_sec = time_to_seconds(START_TIME)
end_sec = time_to_seconds(END_TIME)
window_hours = (end_sec - start_sec) / 3600.0

# Join stop_times with service_id
st = stop_times.merge(trips[["trip_id", "service_id"]], on="trip_id", how="left")

# Filter to active services
st = st[st["service_id"].isin(active_services)].copy()

# Convert arrival times
st["arr_sec"] = st["arrival_time"].apply(time_to_seconds)

# Filter time window
st_peak = st[(st["arr_sec"] >= start_sec) & (st["arr_sec"] <= end_sec)]

# Count arrivals per stop
arrivals = (
    st_peak.groupby("stop_id")
    .size()
    .reset_index(name="Total_Arrivals")
)

# Merge with stops
out = stops[["stop_id", "stop_lon", "stop_lat"]].merge(
    arrivals, on="stop_id", how="left"
)

out["Total_Arrivals"] = out["Total_Arrivals"].fillna(0)

# Convert to hourly frequency Fj
out["Fj"] = out["Total_Arrivals"] / window_hours



In [13]:

out = out.rename(columns={"stop_lon": "lon", "stop_lat": "lat"})
out = out[["stop_id", "Fj", "lon", "lat"]]

out.to_csv(OUTPUT_CSV, index=False)

In [14]:
out.head()


,stop_id,Fj,lon,lat
0,000200403005,8.0,6.113159000000,49.610276000000
1,000200402003,4.5,6.134779000000,49.643399000000
2,000200402004,4.0,6.134546000000,49.647594000000
3,000200304002,9.0,6.142332000000,49.583246000000
4,000200417037,10.0,6.149234000000,49.628363000000


In [15]:
# Top stops by Fj 
print(f"Total stops: {len(out)}")

print(out.nlargest(20, 'Fj').to_string(index=False))

Total stops: 2789
     stop_id    Fj            lon             lat
000200405036 138.0 6.133033000000 49.599066000000
000200404032 130.5 6.139519000000 49.590662000000
000200417050 120.0 6.174766000000 49.636375000000
000200405020 119.5 6.125976000000 49.611504000000
000200405014 108.0 6.126753000000 49.615707000000
000200412031 103.5 6.111496000000 49.576813000000
000200405045 101.0 6.119060000000 49.613509000000
000200412018  97.5 6.113133000000 49.579552000000
000200405013  91.5 6.131646000000 49.608793000000
000200405025  90.0 6.125013000000 49.610454000000
000200412006  83.5 6.115351000000 49.583476000000
000200415002  82.0 6.117916000000 49.597233000000
000200417047  81.5 6.135750000000 49.618368000000
000200405046  79.5 6.136658000000 49.600514000000
000200410003  76.5 6.129933000000 49.627562000000
000200405005  72.5 6.129166000000 49.613535000000
000220402079  71.0 5.955406000000 49.508992000000
000200101002  70.5 6.049868000000 49.621905000000
000200412009  70.0 6.11697500000